# 1. Load modules

In [ ]:
from pathlib import Path
from dataclasses import dataclass, field
from tqdm import tqdm
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.spatial import cKDTree
from sklearn.covariance import MinCovDet
from scipy import stats
from statsmodels.stats.multitest import multipletests 
from scipy.stats import pearsonr
from sklearn.metrics import r2_score

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import altair as alt
alt.data_transformers.disable_max_rows()

from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader

plt.style.use("../../config/DIT_HAP.mplstyle")
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
AX_WIDTH, AX_HEIGHT = plt.rcParams['figure.figsize']

# 2. Configuration

In [ ]:
def min_max_normalization(values: np.ndarray, min_value: float, max_value: float) -> np.ndarray:
    """Normalize values to the range [0, 1] using min-max normalization."""
    if max_value - min_value == 0:
        return np.zeros_like(values)  # Avoid division by zero
    normalized = (values - min_value) / (max_value - min_value)
    return normalized

In [ ]:
@dataclass
class config:

    pombase_version: str = '2025-10-01'
    pombase_dir: Path = Path('/data/c/yangyusheng_optimized/DIT_HAP_pipeline/resources/pombase_data')
    cluster_result_file: Path = Path('/data/c/yangyusheng_optimized/DIT_HAP_pipeline/results/HD_DIT_HAP_generationRAW/18_gene_level_clustering/kmeans_cluster_result.tsv')
    gRNA_result_file: Path = Path('/data/c/yangyusheng_optimized/DIT_HAP_pipeline/resources/260127-all_genes_order1_gRNA_HDdata_fitted_parameters.tsv')
    output_dir: Path = Path('/data/c/yangyusheng_optimized/DIT_HAP_pipeline/results/HD_DIT_HAP_generationRAW/25_complex_analysis')
    DIT_HAP_generations: dict = field(
        default_factory=lambda: {
            "YES0": 0.0,
            "YES1": 2.352,
            "YES2": 5.588,
            "YES3": 9.104,
            "YES4": 12.48,
        })
    gRNA_generations: dict = field(default_factory=lambda: {
        "M_G0Tet": 0.0,
        "M_YES1_Tet": 3.703,
        "M_YES2_Tet": 6.816,
        "M_YES3_Tet": 10.292,
        "M_YES4_Tet": 13.594,
        "M_YES5_Tet": 16.723
    })

    def __post_init__(self):
        self.obo_file = self.pombase_dir / self.pombase_version / "ontologies_and_associations" / "go-basic.obo"
        self.gaf_file = self.pombase_dir / self.pombase_version / "ontologies_and_associations" / "gene_ontology_annotation.gaf.tsv"
        macrocomplex_file = self.pombase_dir / self.pombase_version / "ontologies_and_associations" / "macromolecular_complex_annotation.tsv"
        self.macrocomplex_df: pd.DataFrame = pd.read_csv(macrocomplex_file, sep='\t').rename(columns={"systematic_id": "Systematic ID", "symbol": "Name"})
        self.cluster_result_df: pd.DataFrame = pd.read_csv(self.cluster_result_file, sep='\t')
        self.gRNA_result_df: pd.DataFrame = pd.read_csv(self.gRNA_result_file, sep='\t').rename(columns={"gene_name": "Name"})

        self.cluster_result_df["normalized_um"] = min_max_normalization(self.cluster_result_df["um"].values, 0, 1)
        self.cluster_result_df["normalized_lam"] = min_max_normalization(self.cluster_result_df["lam"].values, 0, 10)
        self.gRNA_result_df["normalized_um"] = min_max_normalization(self.gRNA_result_df["um"].values, 0, 1)
        self.gRNA_result_df["normalized_lam"] = min_max_normalization(self.gRNA_result_df["lam"].values, 0, 14)

        self.DIT_HAP_with_gRNA_df: pd.DataFrame = self.cluster_result_df[["Systematic ID", "Name", "RevisedDeletion_essentiality", "revised_cluster", "um", "lam", "normalized_um", "normalized_lam"]].merge(
            self.gRNA_result_df[["Systematic ID", "Name", "um", "lam", "normalized_um", "normalized_lam"]],
            on=["Systematic ID", "Name"],
            how="outer",
            suffixes=('_DITHAP', '_gRNA')
        )

        self.macrocomplex: pd.DataFrame = self.macrocomplex_df[["complex_term_id", "GO_term_name", "Systematic ID", "Name"]].drop_duplicates().merge(
            self.cluster_result_df[["Systematic ID", "RevisedDeletion_essentiality", "revised_cluster", "um", "lam", "normalized_um", "normalized_lam"]], 
            on="Systematic ID", 
            how="left"
        ).merge(
            self.gRNA_result_df[["Systematic ID", "um", "lam", "normalized_um", "normalized_lam"]], 
            on="Systematic ID", 
            how="left", 
            suffixes=('_DITHAP', '_gRNA')
        )

        DIT_HAP_curves = self.cluster_result_df[["Systematic ID", "Name", "RevisedDeletion_essentiality", "revised_cluster"] + list(self.DIT_HAP_generations.keys())].copy()
        DIT_HAP_curves = DIT_HAP_curves.set_index(
            ["Systematic ID", "Name", "RevisedDeletion_essentiality", "revised_cluster"]
        ).rename_axis("Timepoint", axis=1).stack().to_frame("LFC").reset_index()
        DIT_HAP_curves["Generation"] = DIT_HAP_curves["Timepoint"].map(self.DIT_HAP_generations)

        gRNA_curves = self.gRNA_result_df[["Systematic ID", "Name"] + list(self.gRNA_generations.keys())].copy()
        gRNA_curves = gRNA_curves.set_index(
            ["Systematic ID", "Name"]
        ).rename_axis("Timepoint", axis=1).stack().to_frame("LFC").reset_index()
        gRNA_curves["Generation"] = gRNA_curves["Timepoint"].map(self.gRNA_generations)

        self.DIT_HAP_curves = DIT_HAP_curves
        self.gRNA_curves = gRNA_curves

        self.output_dir.mkdir(parents=True, exist_ok=True)

cfg = config()

# 3. Preparation

## Load Whole GO Ontology

In [ ]:
def load_GO_data(obo_file: str | Path, gaf_file: str | Path):
    """
    Load GO ontology and gene associations.
    
    Parameters:
    -----------
    obo_file : str | Path
        Path to the OBO file containing ontology definitions
    gaf_file : str
        Path to the GAF file containing gene associations
        
    Returns:
    --------
    Tuple[GODag, Dict[str, Dict[str, Set[str]]]]
        GO ontology DAG and namespace-to-associations dictionary
    """
    try:
        # Load GO ontology
        print(f"Loading GO ontology from: {obo_file}")
        godag = GODag(str(obo_file), optional_attrs=['def', 'relationship'], load_obsolete=False)
        print(f"Loaded {len(godag)} GO terms")
        
        # Load gene associations
        print(f"Loading gene associations from: {gaf_file}")
        gaf_reader = GafReader(str(gaf_file), godag = godag)
        
        # Group associations by namespace
        ns2assoc = gaf_reader.get_ns2assc(godag=godag)
            
        return godag, gaf_reader, ns2assoc
        
    except Exception as e:
        print(f"Error loading GO data: {e}")
        return None, None, None


# Load GO data and create comprehensive mapping
print("=" * 60)
print("LOADING GO DATA AND CREATING COMPREHENSIVE MAPPING")
print("=" * 60)

# Load GO ontology and gene associations
godag, gaf_reader, ns2assoc = load_GO_data(cfg.obo_file, cfg.gaf_file)
gene2go = gaf_reader.get_id2gos_nss(propagate_counts=True, relationships={'part_of', 'is_a'}, load_obsolete=False)
go2genes = gaf_reader.get_id2gos_nss(go2geneids=True, propagate_counts=True, relationships={'part_of', 'is_a'}, load_obsolete=False)

In [ ]:
def go_details(go_id: str, go_dag: GODag) -> dict:
    """
    Get the details of a GO term from the GO DAG.
    """
    go_term = go_dag[go_id]

    return {
        "id": go_id,
        "name": go_term.name,
        "namespace": go_term.namespace,
        "definition": go_term.defn,
        "level": go_term.level,
        "depth": go_term.depth,
    }

## Plot function

In [ ]:
def plot_given_subsets_no_merging(sub_df: pd.DataFrame, color_col: str, cfg: config, title: str):
    # DIT-HAP plot
    alt_chart_dithap = alt.Chart(sub_df).mark_point(filled=True, size=100).encode(
        x=alt.X("um_DITHAP", scale=alt.Scale(domain=(-0.3, 1.8))),
        y=alt.Y("lam_DITHAP", scale=alt.Scale(domain=(-1, 10))),
        color=alt.Color(color_col),
        tooltip=sub_df.columns.tolist()
    ).properties(
        title="DIT-HAP"
    )

    # gRNA plot
    alt_chart_gRNA = alt.Chart(sub_df).mark_point(filled=True, size=100).encode(
        x=alt.X("um_gRNA", scale=alt.Scale(domain=(-0.3, 1.8))),
        y=alt.Y("lam_gRNA", scale=alt.Scale(domain=(-1, 14))),
        color=alt.Color(color_col),
        tooltip=sub_df.columns.tolist()
    ).properties(
        title="gRNA"
    )

    # DIT-HAP curves
    sub_DIT_HAP_curves = sub_df.merge(
        cfg.DIT_HAP_curves[["Systematic ID", "Timepoint", "LFC", "Generation"]],
        on="Systematic ID",
        how="left"
    )

    alt_curve_DIT_HAP = alt.Chart(sub_DIT_HAP_curves).mark_line().encode(
        x=alt.X("Generation", scale=alt.Scale(domain=(-0.5, 14))),
        y=alt.Y("LFC", scale=alt.Scale(domain=(-2, 9))),
        color=alt.Color(color_col),
        detail="Systematic ID",
        tooltip=sub_DIT_HAP_curves.columns.tolist()
    ).properties(
        title="DIT-HAP Curves"
    )

    # gRNA curves
    sub_gRNA_curves = sub_df.merge(
        cfg.gRNA_curves[["Systematic ID", "Timepoint", "LFC", "Generation"]],
        on="Systematic ID",
        how="left"
    )

    alt_curve_gRNA = alt.Chart(sub_gRNA_curves).mark_line().encode(
        x=alt.X("Generation", scale=alt.Scale(domain=(-0.5, 18))),
        y=alt.Y("LFC", scale=alt.Scale(domain=(-2, 14))),
        color=alt.Color(color_col),
        detail="Systematic ID",
        tooltip=sub_gRNA_curves.columns.tolist()
    ).properties(
        title="gRNA Curves"
    )

    return alt_chart_dithap, alt_chart_gRNA, alt_curve_DIT_HAP, alt_curve_gRNA

def plot_given_subsets_with_selector(sub_df: pd.DataFrame, color_col: str, cfg: config, title: str, selector):
    alt_chart_dithap, alt_chart_gRNA, alt_curve_DIT_HAP, alt_curve_gRNA = plot_given_subsets_no_merging(sub_df, color_col, cfg, title)

    # Apply the selection to the DIT-HAP plot
    chart_DITHAP = alt_chart_dithap.add_params(selector).encode(
        color=alt.Color("complex", legend=alt.Legend(title=title, labelLimit=0)),
        opacity=alt.condition(selector, alt.value(1), alt.value(0))
    ).transform_filter(
        selector
    )

    # Apply the selection to the gRNA plot
    chart_gRNA = alt_chart_gRNA.add_params(selector).encode(
        color=alt.Color("complex", legend=alt.Legend(title=title, labelLimit=0)),
        opacity=alt.condition(selector, alt.value(1), alt.value(0))
    ).transform_filter(
        selector
    )

    # Apply the selection to the DIT-HAP curves
    curve_DIT_HAP = alt_curve_DIT_HAP.add_params(selector).encode(
        color=alt.Color("complex", legend=alt.Legend(title=title, labelLimit=0)),
        opacity=alt.condition(selector, alt.value(1), alt.value(0))
    ).transform_filter(
        selector
    )

    # Apply the selection to the gRNA curves
    curve_gRNA = alt_curve_gRNA.add_params(selector).encode(
        color=alt.Color("complex", legend=alt.Legend(title=title, labelLimit=0)),
        opacity=alt.condition(selector, alt.value(1), alt.value(0))
    ).transform_filter(
        selector
    )

    # Combine the charts into a single visualization
    combined_chart = alt.vconcat(alt.hconcat(chart_DITHAP, chart_gRNA), alt.hconcat(curve_DIT_HAP, curve_gRNA))

    return combined_chart

def plot_given_subsets(sub_df: pd.DataFrame, color_col: str, cfg: config, title: str):
    alt_chart_dithap, alt_chart_gRNA, alt_curve_DIT_HAP, alt_curve_gRNA = plot_given_subsets_no_merging(sub_df, color_col, cfg, title)
    legend_selector = alt.selection_point(fields=["complex"], bind="legend")
    chart_DITHAP = alt_chart_dithap.add_params(legend_selector).encode(
        color=alt.Color("complex", legend=alt.Legend(title=title, labelLimit=0)),
        opacity=alt.condition(legend_selector, alt.value(1), alt.value(0))
    )

    chart_gRNA = alt_chart_gRNA.add_params(legend_selector).encode(
        color=alt.Color("complex", legend=alt.Legend(title=title, labelLimit=0)),
        opacity=alt.condition(legend_selector, alt.value(1), alt.value(0))
    )

    curve_DIT_HAP = alt_curve_DIT_HAP.add_params(legend_selector).encode(
        color=alt.Color("complex", legend=alt.Legend(title=title, labelLimit=0)),
        opacity=alt.condition(legend_selector, alt.value(1), alt.value(0))
    )

    curve_gRNA = alt_curve_gRNA.add_params(legend_selector).encode(
        color=alt.Color("complex", legend=alt.Legend(title=title, labelLimit=0)),
        opacity=alt.condition(legend_selector, alt.value(1), alt.value(0))
    )

    combined_chart = alt.vconcat(alt.hconcat(chart_DITHAP, chart_gRNA), alt.hconcat(curve_DIT_HAP, curve_gRNA))

    return combined_chart

In [ ]:
def plot_complexes(cfg: config, selected_components: dict, subcomplex_df: pd.DataFrame, title: str, file_name: str):
    fig, ax = plt.subplots(figsize=(AX_WIDTH, AX_HEIGHT))

    x_all = cfg.DIT_HAP_with_gRNA_df["um_DITHAP"].values
    y_all = cfg.DIT_HAP_with_gRNA_df["lam_DITHAP"].values
    ax.scatter(x_all, y_all, color="lightgray", s=100)
    for subcomplex, color in selected_components.items():
        sub_df = subcomplex_df.query("complex == @subcomplex")
        x_sub = sub_df["um_DITHAP"].values
        y_sub = sub_df["lam_DITHAP"].values
        ax.scatter(x_sub, y_sub, color=color, s=100, label=subcomplex)
    ax.set_xlabel("DR")
    ax.set_ylabel("DL")
    ax.set_xlim(-0.15, 1.5)
    ax.set_title(title)
    # ax.legend(loc="best")
    plt.tight_layout()
    plt.savefig(cfg.output_dir / file_name, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

# 4. Different parts of a module

## Functions

In [ ]:
def check_missing_subunits(complex: dict, cfg: config) -> pd.DataFrame:
    """Given a dictionary of complexes and their subunits, check which subunits are missing from the DIT-HAP data and return a combined DataFrame of the subunits that are present."""
    dfs = []
    for subcomplex, subunits in complex.items():
        sub_df = cfg.DIT_HAP_with_gRNA_df.query("Name in @subunits").copy()
        sub_df["complex"] = subcomplex
        dfs.append(sub_df)
        not_in_DIT_HAP = set(subunits) - set(sub_df["Name"])
        if not_in_DIT_HAP:
            print(f"Subunits in {subcomplex} not found in DIT-HAP data: {not_in_DIT_HAP}")
        
    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df



## Cytoplasmic translation

In [ ]:
cytoplasmic_translations = {
    "cytoplasmic translation initiation": [
        "dhx29", "eif21", "gcn3", "rli1", "sui1", "sum3", "tif1", "tif11", "tif1102", "tif211", "tif222", "tif223", "tif224", "tif225", "tif45", "tif452", "tif471", "tif52", "SPCC825.01"
    ],
    "cytoplasmic aminoacyl-tRNA synthetase": [
        "nrs1","drs1","gus1","irs1","lrs1","krs1","rar1","frs2","frs1","prs1","srs1","trs1","wrs1","yrs1","vrs1","ala1","msr1","SPAC29E6.06c","grs1","hrs1"
    ],
    "single-copy ribosomal protein": ["rpl14","rpl13","rps13","rps29","rps3","rpl8","rpp0","rps2","rpl35",]
}

cytoplasmic_translations_df = check_missing_subunits(cytoplasmic_translations, cfg)
cytoplasmic_translations_chart = plot_given_subsets(cytoplasmic_translations_df, "complex", cfg, "Kinetochore")
cytoplasmic_translations_chart#.save(cfg.output_dir / "cytoplasmic_translations_chart.html")

## Kinetochore

In [ ]:
kinetochore = {
    "CENP-A (histone variant)": ["cnp1"],
    "mis6-sim4 complex (inner kinetochore)": ["cnl2", "fta1", "fta2", "fta3", "fta4", "fta6", "fta7", "mal2", "mis15", "mis17", "mis6", "sim4"],
    "CENP-T-W-S-X complex (inner kinetochore)": ["cnp20", "wip1", "mhf1", "mhf2"],
    "Knl1/Spc105 complex (outer kinetochore)": ["spc7", "sos7"],
    "MIS12/MIND complex (outer kinetochore)": ["mis12", "mis13", "mis14", "nnf1"],
    "Ndc80 complex (outer kinetochore)": ["ndc80", "spc25", "spc24", "nuf2"],
    "DASH complex (outer kinetochore)": ["ask1", "dad1", "dad2", "dad3", "dad4", "dad5", "dam1", "duo1", "spc19", "spc34"],
    "KMN complex (outer kinetochore)": ["spc7", "sos7", "mis12", "mis13", "mis14", "nnf1", "ndc80", "spc25", "spc24", "nuf2"]
}

kinetochore_df = check_missing_subunits(kinetochore, cfg)
kinetochore_chart = plot_given_subsets(kinetochore_df, "complex", cfg, "Kinetochore")
kinetochore_chart#.save(cfg.output_dir / "kinetochore_chart.html")

In [ ]:
selected_kinetochore_components = {
    "CENP-A (histone variant)": "#4F609C",
    "mis6-sim4 complex (inner kinetochore)": "#9D343C",
    "CENP-T-W-S-X complex (inner kinetochore)": "#966729",
    "KMN complex (outer kinetochore)": "#356920",
    "DASH complex (outer kinetochore)": "#67448D"
}

plot_complexes(cfg, selected_kinetochore_components, kinetochore_df, "Kinetochore Complexes", "kinetochore_complex_DIT_HAP_space.svg")

## mitochondrion

In [ ]:
mitochondrion = {
    "TOM complex": ["tom20", "tom22", "tom40", "tom5", "tom6", "tom7", "tom70"],
    "TIM23 mitochondrial import inner membrane translocase complex": ["mgr2", "tim17", "tim21", "tim23", "tim50"],
    "TIM22 mitochondrial import inner membrane insertion complex": ["tim22", "tim54"],
    "mitochondrial intermembrane space chaperone complex": ["tim10", "tim13", "tim8", "tim9"],
    "SAM complex": ["mdm10", "mtx1", "mtx2", "sam50"],
    "PAM complex, Tim23 associated import motor": ["mge1", "pam16", "pam17", "pam18", "ssc1", "tim44"],
    "MIM complex": ["mim1", "mim2"],
    "mitochondrial processing peptidase complex": ["qcr1", "mas2"],
    "mitochondrial intermediate peptidase": ["oct1"],
    "mitochondrial inner membrane peptidase complex": ["mmp1", "mmp2"],
    "mitochondrial rhomboid family peptidase": ["rbd1"],
    "MICOS complex": ["mic10", "mic19", "mic26", "mic60"],
    "ERMES complex": ["gem1", "mdm10", "mdm12", "mdm34", "mmm1"],
    "pyruvate decarboxylation to acetyl-CoA": ["dld1", "lat1", "pda1", "pdb1", "pdx1", "pkp1", "ptc5"],
    "protein import into mitochondrial intermembrane space": ["hot13", "hot15", "mcp60"],
    "protein import into mitochondrial matrix (PAM + TIM23 + TOM)": ["mcp60", "mge1", "mgr2", "mom14", "pam16", "pam17", "pam18", "ssc1", "tim17", "tim21", "tim23", "tim44", "tim50", "tom20", "tom22", "tom5", "tom6", "tom7", "xdj1", "zim17"], # PAM + TIM23 + TOM
    "protein insertion into mitochondrial inner membrane (TIM22 + TIM chaperones)": ["mgr2", "tim10", "tim13", "tim22", "tim54", "tim8", "tim9"], # TIM22 + TIM chaperones
    "protein insertion into mitochondrial inner membrane from matrix": ["bcs1", "cox18", "mba1", "oxa101", "oxa102"], # OXA complex
    "protein insertion into mitochondrial outer membrane (MIM + SAM + TOM7)": ["mdm10", "mim1", "mim2", "mtx1", "mtx2", "sam50", "tom7", "SPAC4H3.01", "SPBC3E7.11c"], # MIM + SAM + TOM7
    "mitochondrial translation": [
        "cbp7", "cox1101", "cox1102", "msr1", "SPAC29E6.06c", "grs1", "hrs1", "tac1", "gfm1", "ppr10",
        "gta1", "gta2", "gta3", "mse1", "ism1", "SPAC4G8.09", "msk1", "mdm38", "msm1", "fmt1",
        "mtq1", "pth4", "pth1", "pth3", "msf1", "ppr4", "SPBC24C6.03", "mrpl1", "mrp11", "mrpl19",
        "mrpl12", "mrpl23", "mrpl38", "mrpl10", "mrpl16", "mrpl8", "img1", "rml2", "aco2", "mrpl22",
        "mrp20", "mrpl40", "mrp7", "mrpl24", "mrpl4", "mrpl9", "mrpl33", "tam9", "mrpl32", "mrpl39",
        "mrx14", "new15", "rtc6", "mrpl35", "yml6", "mrpl28", "mrpl27", "mrpl51", "mrpl3", "mrpl17",
        "img2", "mrpl7", "mrpl50", "mug178", "mrp51", "mrpl44", "mrpl37", "mrpl15", "mrpl20", "mrpl25",
        "mrpl6", "mrpl31", "mhr1", "SPCC777.17c", "mrp49", "rsm10", "mrps18", "mrps12", "sws2",
        "mrp2", "mrps28", "mrps16", "mrps17", "rsm18", "rsm19", "mrp4", "mrp21", "rsm25", "mrps26",
        "rsm23", "var1", "rsm27", "rsm24", "mrp10", "cox24", "nam9", "fyv4", "SPBC3H7.04", "SPBC16A3.14",
        "bot1", "snr1", "mrp5", "mrp17", "rsm7", "mrps8", "mrps9", "rrf1", "gfm2", "dia4",
        "SPAC24C9.09", "mrh5", "tsf1", "tuf1", "guf1", "mti2", "mti3", "mrf1", "cbp8", "mtf2",
        "sls1", "mpa1", "SPCC576.06c", "vrs2"
    ],
    "iron-sulfur cluster": ["frp1","fip1","fio1","fet4","mmt1","fxn1","nfs1","isd11","isu1","arh1","etp1","ssc1","jac1","grx5","isa1","isa2","iba57","atm1","dre2","tah18","SPAC806.02c","nar1","mms19","rli1","grx4","fra2","fep1","php4"],
    "iron preprocessing": ["frp1", "fio1", "fip1", "fet4", "mmt1", "fxn1"],
    "sulfur preprocessing": ["nfs1", "isd11"],
    "2Fe-2S cluster assembly": ["arh1", "etp1", "ssc1", "jac1", "isu1", "grx5"],
    "2Fe-2S cluster transfer": ["atm1"],
    "mitochondrial 4Fe-4S cluster assembly": ["isa1", "isa2", "iba57"],
    "cytosolic and nuclear 4Fe-4S cluster assembly": ["tah18", "dre2", "SPAC806.02c", "nar1", "mms19", "rli1"],
    "iron regulation": ["grx4", "fra2", "fep1", "php4"],
    "TCA cycle": ["aco1", "aco2", "cit1", "fum1", "idh1", "idh2", "idp1", "kgd1", "kgd2", "kgd4", "lsc2", "mdh1", "sca1", "sdh1", "sdh2", "sdh3", "sdh4"],
    "oxidative phosphorylation": ["cob1", "cyc1","cox12","cox1","cox2","cox3","cox4","cox5","cox6","cox13","cox7","cox9","cox8","cyt1","nde1","atp1","atp2","atp16","atp15","tim11","atp17","atp20","atp14","atp18","atp19","atp3","atp6","atp8","atp9","atp7","atp5","atp4","ndi1","inh1","sdh3","sdh2","sdh1","qcr2","rip1","qcr7","qcr8","qcr6","qcr9","qcr10"],
    "mitochondrial electron transport, succinate to ubiquinone": ["sdh1", "sdh2", "sdh3"],
    "mitochondrial electron transport, cytochrome c to oxygen": ["cox1", "cox12", "cox13", "cox2", "cox3", "cox4", "cox5", "cox6", "cox7", "cox8", "cox9", "cyc1"],
    "mitochondrial electron transport, ubiquinol to cytochrome c": ["cob1", "cyc1", "cyt1", "qcr10", "qcr2", "qcr6", "qcr7", "qcr8", "qcr9", "rip1"],
    "proton motive force-driven mitochondrial ATP synthesis": ["atp1", "atp14", "atp15", "atp16", "atp17", "atp18", "atp19", "atp2", "atp20", "atp3", "atp4", "atp5", "atp6", "atp7", "atp8", "atp9", "inh1", "tim11"],
    "mitochondrial electron transport, NADH to ubiquinone": ["nde1", "ndi1"],
    "mitochondrial DNA replication": ["pog1", "rnh1"],
    "mitochondrial gene expression": ["cca2","cca1","cbp7","cox1101","cox1102","rnl","nfs1","mrm202","rpm1","trz2","msr1","SPAC29E6.06c","grs1","hrs1","mss116","rpm2","tac1","cob1-I1","gfm1","mrx11","ppr10","ips1","gta1","gta2","gta3","mse1","ism1","SPAC4G8.09","lyr3","cup1","lyr2","msk1","oms1","mdm38","msm1","fmt1","mtq1","pth4","pth1","pth3","msf1","atp25","ppr1","ppr2","ppr3","ppr5","ppr7","ppr8","ppr6","ppr4","SPBC24C6.03","sif2","sif3","mrpl1","mrp11","mrpl19","mrpl12","mrpl23","mrpl38","mrpl10","mrpl16","mrpl8","img1","rml2","aco2","mrpl22","mrp20","mrpl40","mrp7","mrpl24","mrpl4","mrpl9","mrpl33","tam9","mrpl32","mrpl39","mrx14","new15","rtc6","mrpl35","yml6","mrpl28","mrpl27","mrpl51","mrpl3","mrpl17","img2","mrpl7","mrpl50","mug178","mrp51","mrpl44","mrpl37","mrpl15","mrpl20","mrpl25","mrpl6","mrpl31","mhr1","SPCC777.17c","mrp49","rsm10","mrps18","mrps12","sws2","mrp2","mrps28","mrps16","mrps17","rsm18","rsm19","mrp4","mrp21","rsm25","mrps26","rsm23","var1","rsm27","rsm24","mrp10","cox24","nam9","fyv4","SPBC3H7.04","SPBC16A3.14","bot1","snr1","mrp5","mrp17","rsm7","mrps8","mrps9","rrf1","gfm2","rpo41","mtf1","rnpB","mrm1","dia4","SPAC24C9.09","cmb1","mrh5","tsf1","tuf1","guf1","mti2","mti3","mrf1","cbp8","mtf2","sls1","mpa1","rrg8","pgp1","sua5","mss1","slm3","SPCC576.06c","vrs2","rns","trm5"],
}

mitochondrion_df = check_missing_subunits(mitochondrion, cfg)
mitochondrion_chart = plot_given_subsets(mitochondrion_df, "complex", cfg, "Mitochondrion")
mitochondrion_chart

In [ ]:
selected_mitochondrial_subcomplexes = {
    "cytosolic and nuclear 4Fe-4S cluster assembly": "#9D343C",
    "protein import into mitochondrial matrix (PAM + TIM23 + TOM)": "#67448D",
    # "mitochondrial processing peptidase complex": "#966729",
    "mitochondrial translation": "#4F609C",
    "oxidative phosphorylation": "#356920",
}

plot_complexes(cfg, selected_mitochondrial_subcomplexes, mitochondrion_df, "Mitochondrion", "mitochondrial_complexes_DIT_HAP_space.svg")


## vesicle

In [ ]:
vesicle = {
    "SNARE complex": ["bet1", "bos1", "fsv1", "gos1", "pep12", "psy1", "sec20", "sec22", "sec9", "sed5", "sft1", "syb1", "tlg1", "tlg2", "ufe1", "use1", "vsl1", "vti1", "ykt6"],
    "exocyst complex": ["exo70", "exo8", "sec3", "sec5", "sec6", "sec8", "sec10", "sec15"],
    "TRAPP complex": ["bet3", "bet5", "trs23", "trs31", "trs33", "trs65", "trs120", "trs130", "trs8501", "trs8502", "tca17"],
}

vesicle_df = check_missing_subunits(vesicle, cfg)
vesicle_chart = plot_given_subsets(vesicle_df, "complex", cfg, "Vesicle Trafficking")
vesicle_chart

In [ ]:
selected_vesicle_components = {
    "SNARE complex": "#9D343C",
    "exocyst complex": "#423B64",
    "TRAPP complex": "#356920"
}

plot_complexes(cfg, selected_vesicle_components, vesicle_df, "Vesicle Complexes", "vesicle_complex_DIT_HAP_space.svg")

## vacuolar ATPase

In [ ]:
vacuolar_ATPase = {
    "vacuolar proton-transporting V-type ATPase, V0 domain": ["vma11", "vma16", "vma3", "vma6", "vma9", "vph1", "SPAC2F3.18c"],
    "vacuolar proton-transporting V-type ATPase, V1 domain": ["vma1", "vma10", "vma13", "vma2", "vma4", "vma5", "vma7", "vma8"]
}

vacuolar_ATPase_df = check_missing_subunits(vacuolar_ATPase, cfg)
vacuolar_ATPase_chart = plot_given_subsets(vacuolar_ATPase_df, "complex", cfg, "Vacuolar ATPase")
vacuolar_ATPase_chart

# 5. Coherent analysis of different modules

## 5.1 Coherence functions



In [ ]:
def geometric_median(X: np.ndarray, epsilon: float = 1e-5) -> np.ndarray:
    """Compute the geometric median of a set of points using Weiszfeld's algorithm."""
    # Initialize the center y as the component-wise median of the points in X
    y = np.median(X, axis=0)
    
    while True:
        # Calculate the distance from each point in X to the current center y
        distances = cdist(X, [y]).flatten()
        # To avoid division by zero, replace any zero distances with a small value
        distances = np.clip(distances, a_min=epsilon, a_max=None)
        # Calculate weights as the inverse of the distances
        weights = 1.0 / distances
        # Update the center y as the weighted average of the points in X
        y_next = np.average(X, axis=0, weights=weights)
        # Check for convergence: if the change in y is less than epsilon, break the loop
        if np.linalg.norm(y - y_next) < epsilon:
            break
        y = y_next
        
    return y

def average_knn_distance(X: np.ndarray, k: int = 2, method: str = "mean") -> float:
    """ Calculate the average distance to the k nearest neighbors for each point in X using a KD-Tree for efficient querying."""
    n_samples = X.shape[0]
    
    # Robustness check: if the number of samples n is less than or equal to 1, return 0.0 since we cannot compute distances
    if n_samples <= 1:
        return 0.0
        
    actual_k = min(k, n_samples - 1)
    tree = cKDTree(X)
    distances, _ = tree.query(X, k=actual_k + 1)
    
    if actual_k == 1:
        knn_distances = distances[:, 1]
    else:
        knn_distances = distances[:, 1:]
    if method == "mean":
        return float(np.mean(knn_distances))
    elif method == "median":
        return float(np.median(knn_distances))
    else:
        raise ValueError("Invalid method. Choose from 'mean' or 'median'.")

def distance_to_centroid(X: np.ndarray, centroid: np.ndarray | None = None, method: str = "median") -> float | np.ndarray:
    """Calculate the distance from each point in X to the geometric median."""
    if centroid is None:
        centroid = geometric_median(X)
    distances = cdist(X, [centroid]).flatten()
    if method == "mean":
        return float(np.mean(distances))
    elif method == "median":
        return float(np.median(distances))
    elif method == "max":
        return float(np.max(distances))
    elif method == "both":
        return np.array([np.mean(distances), np.median(distances)])
    elif method == "all":
        return np.array([np.mean(distances), np.median(distances), np.max(distances)])
    else:
        raise ValueError("Invalid method. Choose from 'mean', 'median', 'max', 'both' or 'all'.")

def pairwise_distance(X: np.ndarray, method: str = "median", k_nn: int = 3) -> float | np.ndarray:
    """Calculate the pairwise distances between points in X."""
    pairwise_distance = pdist(X)
    if method == "mean":
        return float(np.mean(pairwise_distance))
    elif method == "median":
        return float(np.median(pairwise_distance))
    elif method == "max":
        return float(np.max(pairwise_distance))
    elif method == "knn":
        return average_knn_distance(X, k=k_nn, method="mean")
    elif method == "both":
        return np.array([np.mean(pairwise_distance), np.median(pairwise_distance)])
    elif method == "all":
        return np.array([np.mean(pairwise_distance), np.median(pairwise_distance), np.max(pairwise_distance), average_knn_distance(X, k=k_nn)])
    else:
        raise ValueError("Invalid method. Choose from 'mean', 'median', 'max', 'both' or 'all'.")

def coherence_metrics(X: np.ndarray) -> dict:
    """Calculate coherence metrics for a set of points in X."""
    centroid = geometric_median(X)
    distance_to_centroid_stats = distance_to_centroid(X, method="both")
    pairwise_stats = pairwise_distance(X, method="all")
    if isinstance(distance_to_centroid_stats, np.ndarray):
        mean_distance_to_centroid, median_distance_to_centroid = distance_to_centroid_stats.tolist()
    else:
        mean_distance_to_centroid = float(distance_to_centroid_stats)
        median_distance_to_centroid = float(distance_to_centroid_stats)
    if isinstance(pairwise_stats, np.ndarray):
        mean_pairwise_distance, median_pairwise_distance, max_pairwise_distance, mean_knn_distance = pairwise_stats.tolist()
    else:
        mean_pairwise_distance = float(pairwise_stats)
        median_pairwise_distance = float(pairwise_stats)
        max_pairwise_distance = float(pairwise_stats)
        mean_knn_distance = float(pairwise_stats)
    return {
        "centroid_x": centroid[0],
        "centroid_y": centroid[1],
        "mean_distance_to_centroid": mean_distance_to_centroid,
        "median_distance_to_centroid": median_distance_to_centroid,
        "mean_pairwise_distance": mean_pairwise_distance,
        "median_pairwise_distance": median_pairwise_distance,
        "max_pairwise_distance": max_pairwise_distance,
        "mean_knn_distance": mean_knn_distance
    }

def compute_mpd_zscore(X: np.ndarray, bg: np.ndarray, n_permutations: int = 1000, random_state: int | None = None) -> float:
    """Compute the z-score of the mean pairwise distance (MPD) for a set of points in X compared to a null distribution generated by random permutations."""
    rng = np.random.default_rng(random_state)
    observed_mpd = pairwise_distance(X, method="median")
    n_samples = X.shape[0]
    if n_samples <= 1:
        return 0.0, 1
    permuted_mpds = []
    for _ in range(n_permutations):
        permuted_X = rng.choice(bg, size=n_samples, replace=False)
        permuted_mpd = pairwise_distance(permuted_X, method="median")
        permuted_mpds.append(permuted_mpd)
    permuted_mpds = np.array(permuted_mpds)
    mean_permuted_mpd = np.mean(permuted_mpds)
    std_permuted_mpd = np.std(permuted_mpds)
    # p value can be calculated as the proportion of permuted MPDs that are less than or equal to the observed MPD
    p_value = np.mean(permuted_mpds <= observed_mpd)
    if std_permuted_mpd == 0:
        return 0.0, 1
    z_score = (observed_mpd - mean_permuted_mpd) / std_permuted_mpd
    return z_score, p_value

def compute_mcd_zscore(X: np.ndarray, bg: np.ndarray, n_permutations: int = 1000, random_state: int | None = None) -> float:
    """Compute the z-score of the minimal covariance determinant (MCD) for a set of points in X compared to a null distribution generated by random permutations."""
    rng = np.random.default_rng(random_state)
    mcd = MinCovDet().fit(X)
    observed_mcd = mcd.covariance_.diagonal().mean()
    n_samples = X.shape[0]
    if n_samples <= 1:
        return 0.0, 1
    permuted_mcds = []
    for _ in range(n_permutations):
        permuted_X = rng.choice(bg, size=n_samples, replace=False)
        permuted_mcd = MinCovDet().fit(permuted_X)
        permuted_mcd_value = permuted_mcd.covariance_.diagonal().mean()
        permuted_mcds.append(permuted_mcd_value)
    permuted_mcds = np.array(permuted_mcds)
    mean_permuted_mcd = np.mean(permuted_mcds)
    std_permuted_mcd = np.std(permuted_mcds)
    p_value = np.mean(permuted_mcds <= observed_mcd)
    if std_permuted_mcd == 0:
        return 0.0, 1
    z_score = (observed_mcd - mean_permuted_mcd) / std_permuted_mcd
    return z_score, p_value

def compute_distance_zscore(X: np.ndarray, bg: np.ndarray, method: str, n_permutations: int = 1000, random_state: int | None = None) -> float:
    """Compute the z-score of the minimal covariance determinant (MCD) for a set of points in X compared to a null distribution generated by random permutations."""
    rng = np.random.default_rng(random_state)
    match method:
        case "mean_distance_to_centroid":
            observed_distance = distance_to_centroid(X, method="mean")
        case "median_distance_to_centroid":
            observed_distance = distance_to_centroid(X, method="median")
        case "mean_pairwise_distance":
            observed_distance = pairwise_distance(X, method="mean")
        case "median_pairwise_distance":
            observed_distance = pairwise_distance(X, method="median")
        case "max_pairwise_distance":
            observed_distance = pairwise_distance(X, method="max")
        case "mean_knn_distance":
            observed_distance = pairwise_distance(X, method="knn")
        case _:
            raise ValueError("Invalid method. Choose from 'mean_distance_to_centroid', 'median_distance_to_centroid', 'mean_pairwise_distance', 'median_pairwise_distance', 'max_pairwise_distance' or 'mean_knn_distance'.")
    n_samples = X.shape[0]
    if n_samples <= 1:
        return 0.0, 1
    permuted = []
    for _ in range(n_permutations):
        permuted_X = rng.choice(bg, size=n_samples, replace=False)
        match method:
            case "mean_distance_to_centroid":
                permuted_distance = distance_to_centroid(permuted_X, method="mean")
            case "median_distance_to_centroid":
                permuted_distance = distance_to_centroid(permuted_X, method="median")
            case "mean_pairwise_distance":
                permuted_distance = pairwise_distance(permuted_X, method="mean")
            case "median_pairwise_distance":
                permuted_distance = pairwise_distance(permuted_X, method="median")
            case "max_pairwise_distance":
                permuted_distance = pairwise_distance(permuted_X, method="max")
            case "mean_knn_distance":
                permuted_distance = pairwise_distance(permuted_X, method="knn")
            case _:
                raise ValueError("Invalid method. Choose from 'mean_distance_to_centroid', 'median_distance_to_centroid', 'mean_pairwise_distance', 'median_pairwise_distance', 'max_pairwise_distance' or 'mean_knn_distance'.")
        permuted.append(permuted_distance)
    permuted = np.array(permuted)
    mean_permuted = np.mean(permuted)
    std_permuted = np.std(permuted)
    p_value = np.mean(permuted <= observed_distance)
    if std_permuted == 0:
        return 0.0, 1
    z_score = (observed_distance - mean_permuted) / std_permuted
    return z_score, p_value

## 5.2 Filter the non-essential genes

In [ ]:
DR_gt_point3_complexes = cfg.macrocomplex.query("um_DITHAP > 0.3")
DR_gt_point3_genes = cfg.DIT_HAP_with_gRNA_df.query("um_DITHAP > 0.3")

## 5.3 Coherence calculation

In [ ]:
def sub_df_coherence(sub_df: pd.DataFrame, bg_df: pd.DataFrame) -> dict:
    term_id = sub_df["complex_term_id"].iloc[0]
    go_detail = go_details(term_id, godag)
    term_size = len(sub_df)
    covered_genes = sorted(list(set(sub_df["Name"])))
    DIT_HAP_values = sub_df[["normalized_um_DITHAP", "normalized_lam_DITHAP"]].values
    gRNA_values = sub_df[["normalized_um_gRNA", "normalized_lam_gRNA"]].values
    DIT_HAP_bg_values = bg_df[["normalized_um_DITHAP", "normalized_lam_DITHAP"]].values
    gRNA_bg_values = bg_df[["normalized_um_gRNA", "normalized_lam_gRNA"]].dropna().values
    DIT_HAP_coherence = coherence_metrics(DIT_HAP_values)
    gRNA_coherence = coherence_metrics(gRNA_values)
    combined_coherence = go_detail.copy()
    combined_coherence["covered_genes"] = ", ".join(covered_genes)
    combined_coherence["term_size"] = term_size
    for key in DIT_HAP_coherence.keys():
        combined_coherence[f"DIT_HAP_{key}"] = DIT_HAP_coherence[key]
        combined_coherence[f"gRNA_{key}"] = gRNA_coherence[key]
    
    for key in ["mean_distance_to_centroid", "median_distance_to_centroid", "mean_pairwise_distance", "median_pairwise_distance", "max_pairwise_distance", "mean_knn_distance"]:
        DIT_HAP_zscore, DIT_HAP_pvalue = compute_distance_zscore(DIT_HAP_values, DIT_HAP_bg_values, method=key, n_permutations=1000, random_state=42)
        gRNA_zscore, gRNA_pvalue = compute_distance_zscore(gRNA_values, gRNA_bg_values, method=key, n_permutations=1000, random_state=42)
        combined_coherence[f"DIT_HAP_{key}_zscore"] = DIT_HAP_zscore
        combined_coherence[f"DIT_HAP_{key}_pvalue"] = DIT_HAP_pvalue
        combined_coherence[f"gRNA_{key}_zscore"] = gRNA_zscore
        combined_coherence[f"gRNA_{key}_pvalue"] = gRNA_pvalue

    return pd.Series(combined_coherence)

groups = {name: group for name, group in DR_gt_point3_complexes.groupby("GO_term_name") if len(group) >= 3 and len(group) <= 300}
results = []
for name, group in tqdm(groups.items(), desc="Computing coherence"):
    result = sub_df_coherence(group, DR_gt_point3_genes)
    result.name = name
    results.append(result)

complex_coherences = pd.DataFrame(results)
complex_coherences.index.name = "complex"
complex_coherences = complex_coherences.reset_index()

# complex_coherences["corrected_DIT_HAP_mpd_pvalue"] = multipletests(complex_coherences["DIT_HAP_mpd_pvalue"], method="fdr_bh")[1]
# complex_coherences["corrected_gRNA_mpd_pvalue"] = multipletests(complex_coherences["gRNA_mpd_pvalue"], method="fdr_bh")[1]
# complex_coherences["log10_corrected_DIT_HAP_mpd_pvalue"] = -np.log10(complex_coherences["corrected_DIT_HAP_mpd_pvalue"])
# complex_coherences["log10_corrected_gRNA_mpd_pvalue"] = -np.log10(complex_coherences["corrected_gRNA_mpd_pvalue"])

In [ ]:
# complex_coherences.to_csv(cfg.output_dir / "complex_coherence_metrics.csv", index=False)
complex_coherences = pd.read_csv(cfg.output_dir / "complex_coherence_metrics.csv")

## 5.4 Complex coherence visualization and checking

### Complex size distribution

In [ ]:
complex_coherences.hist(["term_size", ], bins=20)
plt.show()
plt.close()

### Visualization of different size complexes

In [ ]:
complex_terms = DR_gt_point3_complexes.groupby("GO_term_name")["Name"].apply(list).to_dict()
complex_terms_df = check_missing_subunits(complex_terms, cfg)
complex_terms_df.to_csv(cfg.output_dir / "complex_terms_DIT_HAP_with_gRNA.csv", index=False)

In [ ]:
def plot_single_complex(sub_df):
    title = sub_df["complex"].iloc[0]
    size = len(sub_df)
    alt_chart_dithap = alt.Chart(sub_df).mark_point(filled=True, size=100).encode(
        x=alt.X("um_DITHAP", scale=alt.Scale(domain=(-0.3, 1.8))),
        y=alt.Y("lam_DITHAP", scale=alt.Scale(domain=(-1, 10))),
        tooltip=sub_df.columns.tolist()
    ).properties(
        title= f"{title} (n={size})"
    )
    return alt_chart_dithap


box_CD_methylation_guide_snoRNP_complex = plot_single_complex(complex_terms_df.query("complex == 'box C/D methylation guide snoRNP complex'"))
delta_DNA_polymerase_complex = plot_single_complex(complex_terms_df.query("complex == 'delta DNA polymerase complex'"))
DNA_replication_preinitiation_complex = plot_single_complex(complex_terms_df.query("complex == 'DNA replication preinitiation complex'"))
mitochondrial_small_ribosomal_subunit = plot_single_complex(complex_terms_df.query("complex == 'mitochondrial small ribosomal subunit'"))

In [ ]:
box_CD_methylation_guide_snoRNP_complex | delta_DNA_polymerase_complex | DNA_replication_preinitiation_complex | mitochondrial_small_ribosomal_subunit

### Visualization of complex coherence

In [ ]:
complex_selector = alt.selection_point(fields=["complex"])

DIT_HAP_coherence = alt.Chart(complex_coherences).mark_circle().encode(
    x=alt.X("DIT_HAP_median_pairwise_distance_zscore"),
    y=alt.Y("gRNA_median_pairwise_distance_zscore"),
    size=alt.Size("term_size", scale=alt.Scale(type="sqrt")),
    tooltip=complex_coherences.columns.tolist()
).add_params(complex_selector).encode(
    opacity=alt.condition(complex_selector, alt.value(0.6), alt.value(0.1))
)

DIT_HAP_vs_gRNA_coherence = alt.Chart(complex_coherences).mark_circle().encode(
    x=alt.X("DIT_HAP_median_pairwise_distance_zscore"),
    y=alt.Y("DIT_HAP_mean_pairwise_distance_zscore"),
    size=alt.Size("term_size", scale=alt.Scale(type="sqrt")),
    tooltip=complex_coherences.columns.tolist()
).add_params(complex_selector).encode(
    opacity=alt.condition(complex_selector, alt.value(0.6), alt.value(0.1))
)

coherence_complex_visualization = plot_given_subsets_with_selector(
    complex_terms_df,
    "complex",
    cfg,
    "Complex Subunits",
    complex_selector
)

((DIT_HAP_coherence & DIT_HAP_vs_gRNA_coherence) | coherence_complex_visualization).interactive()#.save(cfg.output_dir / "complex_coherence_visualization.html", inline=True)

In [ ]:
complex_coherences.to_csv(cfg.output_dir / "complex_coherence_metrics.csv", index=False)

In [ ]:
distance_and_transformations = {
    "mean_distance_to_centroid": "log",
    "median_distance_to_centroid": "log",
    "mean_pairwise_distance": "log",
    "median_pairwise_distance": "log",
    "max_pairwise_distance": "log",
    "mean_knn_distance": "log",
    "mean_distance_to_centroid_zscore": "linear",
    "median_distance_to_centroid_zscore": "linear",
    "mean_pairwise_distance_zscore": "linear",
    "median_pairwise_distance_zscore": "linear",
    "max_pairwise_distance_zscore": "linear",
    "mean_knn_distance_zscore": "linear"
}

fig, axes = plt.subplots(2, 6, figsize=(36, 12))
axes = axes.flatten()
for idx, (column, transformation) in enumerate(distance_and_transformations.items()):
    ax = axes[idx]
    x = complex_coherences["DIT_HAP_" + column]
    y = complex_coherences["gRNA_" + column]
    ax.scatter(x, y, s=complex_coherences["term_size"] * 10, alpha=0.6)
    if transformation == "log":
        ax.set_xscale("log")
        ax.set_yscale("log")
        log_x = np.log1p(x)
        log_y = np.log1p(y)
        PCC = pearsonr(log_x, log_y)[0]
        r2 = r2_score(log_x, log_y)
    else:
        PCC = pearsonr(x, y)[0]
        r2 = r2_score(x, y)
    ax.text(0.05, 0.95, f"PCC: {PCC:.2f}\nR²: {r2:.2f}", transform=ax.transAxes, verticalalignment="top")
    ax.set_title(column)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Create a jointplot where marker size encodes complex term size
sizes = np.sqrt(complex_coherences["term_size"]) * 80
g = sns.jointplot(data=complex_coherences,
                  x="DIT_HAP_median_pairwise_distance_zscore",
                  y="DIT_HAP_mean_pairwise_distance_zscore",
                  kind="scatter",
                  alpha=0.6,
                  height=8,
                  space=0,
                  marginal_ticks=False,
                  joint_kws={"s": sizes})

# Add a custom size legend outside the plot
ax = g.ax_joint

ax.set_xlabel("Median Pairwise Distance Z-score")
ax.set_ylabel("Mean Pairwise Distance Z-score")

ax.plot([-6, 4], [-6, 4], ls="--", c="gray", alpha=0.2)

# Get the color from the scatter plot
scatter_color = ax.collections[0].get_facecolors()[0]

# choose representative term sizes (25th, 50th, 75th, 95th percentiles)
rep_terms = [5, 10, 25, 50]
rep_terms = sorted(set(rep_terms))  # remove duplicates
rep_sizes = np.sqrt(rep_terms) * 80

legend_handles = [
    ax.scatter([], [], s=s, color=scatter_color, alpha=0.6, edgecolors='none') 
    for s in rep_sizes
]
legend_labels = [f"n={t}" for t in rep_terms]

# Place legend outside the plot area
# 添加 scatterpoints=1 参数以确保图例中每个标签只出现一个点
ax.legend(legend_handles, legend_labels, title='Complex size', 
          loc='upper left', bbox_to_anchor=(1.25, 1), frameon=True,
          scatterpoints=1, markerscale=1)

plt.tight_layout()
plt.savefig(cfg.output_dir / "complex_coherence_scatter_with_size_legend.pdf", dpi=300, bbox_inches="tight")
plt.savefig(cfg.output_dir / "complex_coherence_scatter_with_size_legend.svg", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

### Visualization of the complex coherence zscore

In [ ]:
complex_coherences.hist(["DIT_HAP_mean_distance_to_centroid_zscore", "DIT_HAP_median_distance_to_centroid_zscore", "DIT_HAP_mean_pairwise_distance_zscore", "DIT_HAP_median_pairwise_distance_zscore", 
                         "gRNA_mean_distance_to_centroid_zscore", "gRNA_median_distance_to_centroid_zscore", "gRNA_mean_pairwise_distance_zscore", "gRNA_median_pairwise_distance_zscore"
                         ], bins=20, figsize=(20, 10), layout=(2,4))
plt.show()
plt.close()

# Coherent complex visualization

In [ ]:
selected_complexes = {
    "selected_coherent_complexes": [
        "Mis6-Sim4 complex",
        "chaperonin-containing T-complex",
        "mitochondrial large ribosomal subunit",
        "small-subunit processome",
    ],
    "selected_incoherent_complexes": [
        "SAGA complex",
        "eukaryotic 43S preinitiation complex",
        "nuclear pore nuclear basket",
        "nuclear pore outer ring"
    ]
}
color_idx = 0
for selected_type, selected_complexes in selected_complexes.items():
    fig, axes = plt.subplots(2,2,figsize=(12, 12))
    axes = axes.flatten()

    for idx, term in enumerate(selected_complexes):
        subset_df = complex_terms_df.query("complex == @term")
        if len(term) > 30:
            # wrap long titles
            term_elements = term.split(" ")
            sep_idx = len(term_elements) // 2
            term = " ".join(term_elements[:sep_idx]) + "\n" + " ".join(term_elements[sep_idx:])
        ax = axes[idx]
        # all points in light gray
        x_all = cfg.cluster_result_df["um"]
        y_all = cfg.cluster_result_df["lam"]
        ax.scatter(x_all, y_all, color='lightgray', alpha=0.4, rasterized=True)
        # points for given genes
        
        x_subset = subset_df["um_DITHAP"]
        y_subset = subset_df["lam_DITHAP"]

        ax.scatter(x_subset, y_subset, color=COLORS[color_idx], label=f"{term} (n={len(subset_df)})")
        color_idx = color_idx + 1

        ax.set_xlim(-0.3, 1.8)
        ax.set_ylim(-0.5, 14)
        ax.set_xlabel("DR")
        ax.set_ylabel("DL")
        ax.set_title(f"{term} (n={len(subset_df)})")

    plt.tight_layout()
    plt.savefig(cfg.output_dir / f"{selected_type}_DIT_HAP_space.pdf", dpi=300, bbox_inches="tight")
    plt.savefig(cfg.output_dir / f"{selected_type}_DIT_HAP_space.svg", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# 6. Calculate complex level DR DL

In [ ]:
def calculate_complex_level_stats(sub_df):
    complex_size = len(sub_df)
    mean_um_DITHAP = sub_df["um_DITHAP"].mean()
    median_um_DITHAP = sub_df["um_DITHAP"].median()
    mean_um_gRNA = sub_df["um_gRNA"].mean()
    median_um_gRNA = sub_df["um_gRNA"].median()
    mean_lam_DITHAP = sub_df["lam_DITHAP"].mean()
    median_lam_DITHAP = sub_df["lam_DITHAP"].median()
    mean_lam_gRNA = sub_df["lam_gRNA"].mean()
    median_lam_gRNA = sub_df["lam_gRNA"].median()
    return pd.Series({
        "term_size": complex_size,
        "mean_um_DITHAP": mean_um_DITHAP,
        "median_um_DITHAP": median_um_DITHAP,
        "mean_um_gRNA": mean_um_gRNA,
        "median_um_gRNA": median_um_gRNA,
        "mean_lam_DITHAP": mean_lam_DITHAP,
        "median_lam_DITHAP": median_lam_DITHAP,
        "mean_lam_gRNA": mean_lam_gRNA,
        "median_lam_gRNA": median_lam_gRNA
    })

In [ ]:
coherent_complexes = complex_coherences.query("DIT_HAP_median_pairwise_distance_zscore < -1 and DIT_HAP_mean_pairwise_distance_zscore < -1")
coherent_complex_level_stats = complex_terms_df[complex_terms_df["complex"].isin(coherent_complexes["complex"])].groupby("complex").apply(calculate_complex_level_stats)

In [ ]:
alt.Chart(coherent_complex_level_stats.reset_index()).mark_point(filled=True, size=100).encode(
    x=alt.X("mean_um_DITHAP", scale=alt.Scale(domain=(-0.3, 1.8))),
    y=alt.Y("mean_lam_DITHAP", scale=alt.Scale(domain=(-1, 10))),
    size=alt.Size("term_size", scale=alt.Scale(type="sqrt")),
    tooltip=coherent_complex_level_stats.reset_index().columns.tolist()
) | alt.Chart(coherent_complex_level_stats.reset_index()).mark_point(filled=True, size=100).encode(
    x=alt.X("median_um_DITHAP", scale=alt.Scale(domain=(-0.3, 1.8))),
    y=alt.Y("median_lam_DITHAP", scale=alt.Scale(domain=(-1, 10))),
    size=alt.Size("term_size", scale=alt.Scale(type="sqrt")),
    tooltip=coherent_complex_level_stats.reset_index().columns.tolist()
)

In [ ]:
# Create a jointplot where marker size encodes complex term size
sizes = np.sqrt(coherent_complex_level_stats["term_size"]) * 80
g = sns.jointplot(data=coherent_complex_level_stats,
                  x="mean_um_DITHAP",
                  y="mean_lam_DITHAP",
                  kind="scatter",
                  alpha=0.6,
                  height=8,
                  space=0,
                  marginal_ticks=False,
                  joint_kws={"s": sizes})

# Add a custom size legend outside the plot
ax = g.ax_joint

ax.set_xlabel("Complex level DR (mean)")
ax.set_ylabel("Complex level DL (mean)")

# Get the color from the scatter plot
scatter_color = ax.collections[0].get_facecolors()[0]

# choose representative term sizes (25th, 50th, 75th, 95th percentiles)
rep_terms = [5, 10, 25, 50]
rep_terms = sorted(set(rep_terms))  # remove duplicates
rep_sizes = np.sqrt(rep_terms) * 80

legend_handles = [
    ax.scatter([], [], s=s, color=scatter_color, alpha=0.6, edgecolors='none') 
    for s in rep_sizes
]
legend_labels = [f"n={t}" for t in rep_terms]

# Place legend outside the plot area
# 添加 scatterpoints=1 参数以确保图例中每个标签只出现一个点
ax.legend(legend_handles, legend_labels, title='Complex size', 
          loc='upper left', bbox_to_anchor=(1.25, 1), frameon=True,
          scatterpoints=1, markerscale=1)

ax.set_xlim(0, 1.4)
ax.set_ylim(-1, 8)

plt.tight_layout()
plt.savefig(cfg.output_dir / "complex_level_DR_DL_scatter_with_size_legend.pdf", dpi=300, bbox_inches="tight")
plt.savefig(cfg.output_dir / "complex_level_DR_DL_scatter_with_size_legend.svg", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Gene features vs Cluster

In [ ]:
gene_features = pd.read_csv("../../resources/pombe_features/2025-10-01_pombe_coding_gene_protein_features.tsv", sep="\t")
gene_clusters = pd.read_csv("../../results/HD_DIT_HAP_generationRAW/18_gene_level_clustering/all_coding_genes_with_DIT_HAP_clustering.tsv", sep="\t")

feature_and_targets = gene_features.merge(gene_clusters[["Systematic ID", "DR", "DL", "Cluster"]], left_on="gene_systematic_id", right_on="Systematic ID", how="left")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator, FixedLocator, NullLocator
from scipy import stats

# 准备绘图数据 (假设前面已经生成了 feature_and_targets)
plot_data = feature_and_targets.dropna(subset=["DR", "DL", "Cluster"])
plot_data = plot_data[plot_data["Cluster"] < 7]

key_features = [
    "mean_EMM_Proliferating_Cell_RNA_Abundance",
    "mRNA_half_life_minutes",
    "mRNA_synthesis_rate_per_minute",
    "copies_per_cell_EMM_Proliferating_Cell",
    "protein_half_life_minutes",
    "evolutionary_rate"
]

# 为每个特征定义展示的 Title (display label) 和 X轴 Label (unit)
feature_info = {
    "mean_EMM_Proliferating_Cell_RNA_Abundance": {
        "title": "mRNA Abundance", 
        "unit": "Copies / Cell"
    },
    "mRNA_half_life_minutes": {
        "title": "mRNA Half-life", 
        "unit": "Minutes"
    },
    "mRNA_synthesis_rate_per_minute": {
        "title": "mRNA Synthesis Rate", 
        "unit": "Molecules / Minute"
    },
    "copies_per_cell_EMM_Proliferating_Cell": {
        "title": "Protein Copies per Cell", 
        "unit": "Copies / Cell"
    },
    "protein_half_life_minutes": {
        "title": "Protein Half-life", 
        "unit": "Minutes"
    },
    "evolutionary_rate": {
        "title": "Evolutionary Rate", 
        "unit": "Substitutions / Site"
    }
}

# Define colors for each cluster
cluster_colors = [
    "#dd8369", "#6b99df", "#98a64e", "#64af6d", "#a78bd9", 
    "#d57fbd", "#c4954b", "#4bb29c", "#e0788f", "#4aadce"
]

if not plot_data.empty:
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    axes = axes.flatten()
    
    # 定义深灰色，介于 axis 的 gray 和黑色之间
    dark_gray = '#555555'
    
    for i, feature in enumerate(key_features):
        ax = axes[i]
        
        log_transformed_data = [] 
        raw_data_for_test = []    
        cluster_labels = []
        colors = []
        
        for cluster in sorted(plot_data["Cluster"].unique(), reverse=True):
            raw_vals = plot_data[plot_data["Cluster"] == cluster][feature].dropna()
            
            if i != 5: # 前 5 个特征需要 log 转换绘制
                raw_vals = raw_vals[raw_vals > 0] # 过滤掉非正数防止报错
                plot_vals = np.log10(raw_vals)
            else:
                plot_vals = raw_vals # evolutionary_rate 原样画图
                
            if len(raw_vals) > 0:
                raw_data_for_test.append(raw_vals)
                log_transformed_data.append(plot_vals)
                cluster_labels.append(f'{int(cluster)}')
                colors.append(cluster_colors[int(cluster)-1])
        
        if log_transformed_data:
            # 绘制水平 Boxplot，边缘线设置为深灰色，同时强制中位线 capstyle 为 'butt' 截断样式防溢出
            bp = ax.boxplot(log_transformed_data, labels=cluster_labels, patch_artist=True,
                           vert=False, showfliers=False,
                           boxprops=dict(linewidth=1.5, color=dark_gray),
                           medianprops=dict(color=dark_gray, linewidth=2, solid_capstyle='butt'),
                           whiskerprops=dict(linewidth=1.5, color=dark_gray),
                           capprops=dict(linewidth=1.5, color=dark_gray))
            
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_edgecolor(dark_gray) # 强制将被填充 box 的边缘改为深灰色
                patch.set_alpha(0.7)
            
            # --------- 添加相邻组间的 p-value 统计标记 ---------
            for j in range(len(raw_data_for_test) - 1):
                # 使用原始数据做 Mann-Whitney U 检验（最稳健）
                stat, p_val = stats.mannwhitneyu(raw_data_for_test[j], raw_data_for_test[j + 1], alternative='two-sided')
                
                sig = f'p={p_val:.1e}' if p_val < 0.001 else f'p={p_val:.3f}'
                text_color = '#8A2A44' if p_val < 0.05 else 'black'
                font_weight = 'bold' if p_val < 0.05 else 'normal'
                y1, y2 = j + 1 + 0.1, j + 2 - 0.1
                x_bracket, x_text = 1.02, 1.05
                
                # 连线保持黑色
                ax.plot([x_bracket, x_text, x_text, x_bracket], [y1, y1, y2, y2], 
                        transform=ax.get_yaxis_transform(), color='black', lw=1.2, clip_on=False)
                ax.text(x_text + 0.01, (y1 + y2) / 2, sig, transform=ax.get_yaxis_transform(), 
                        ha='left', va='center', fontsize=12, fontweight=font_weight, clip_on=False)
        
        # 格式化和坐标轴标注
        if i == 0:
            ax.set_ylabel('DIT-HAP Cluster', fontsize=14, fontweight='bold')
            
        ax.set_yticklabels(cluster_labels, fontsize=18, fontweight='bold')
        
        # --- 强制隐藏 Y 轴所有的 minor ticks ---
        ax.yaxis.set_minor_locator(NullLocator())
        
        # 映射 Title 和 Unit Label
        ax.set_title(feature_info[feature]["title"], fontsize=16, fontweight='bold', pad=10)
        ax.set_xlabel(feature_info[feature]["unit"], fontsize=14, fontweight='bold')
        
        # 将网格线变淡一些，不要干扰主要视觉
        ax.grid(True, axis='x', which='major', alpha=0.3, linestyle='--')
        
        # 统一步骤：应用 mplstyle 提取定义的 Major / Tick Label 标准
        ax.tick_params(axis='both', which='major', labelsize=18, labelcolor='black', 
                       length=10, width=2, color='gray', direction='out')

        # --------- 核心：模拟真实的 Log Scale 视觉外观 ---------
        if i != 5:
            # 1. 主刻度仅保留严格的整数位（即 1, 2, 3，对应原始值的 10, 100, 1000）
            ax.xaxis.set_major_locator(MultipleLocator(base=1.0))
            
            # 2. 生成对数刻度下的副刻度 (minor ticks)
            # 在绘图数据的常见量级 (-5 到 10 个零) 中插入 log10(2)~log10(9) 的间距
            minor_ticks = []
            for base_val in range(-5, 10):
                for step in range(2, 10):
                    minor_ticks.append(base_val + np.log10(step))
            
            # 设置副刻度 
            ax.xaxis.set_minor_locator(FixedLocator(minor_ticks))
            
            # 3. 映射回来：让主要刻度的数字显示变成 10 / 100 / 1000 
            def format_log_ticks(val, pos):
                real_val = 10**val
                if real_val >= 1000 or real_val <= 0.01: # 包括极小数字也用科学计数法
                    return f"$10^{{{int(val)}}}$"
                elif real_val >= 1:
                    return f"{int(real_val)}"
                else:
                    # 对于 0.1 类似的值特殊处理显示完整浮点以防精度丢失
                    return f"{real_val:.1f}" if real_val == 0.1 else f"{real_val:.2g}"

            ax.xaxis.set_major_formatter(FuncFormatter(format_log_ticks))
            
            # ========= 应用 mplstyle 提取定义的 Minor 标准 =========
            ax.tick_params(axis='x', which='minor', length=5, width=1, color='gray', direction='out')
            ax.tick_params(axis='x', which='major', length=10, width=2, color='gray', direction='out')
    
    plt.tight_layout(h_pad=2, w_pad=3) 
    plt.subplots_adjust(top=0.93)
    
    if 'cfg' in locals() and hasattr(cfg, 'output_dir'):
        plt.savefig(cfg.output_dir / "key_features_vs_cluster_boxplot.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(cfg.output_dir / "key_features_vs_cluster_boxplot.svg", dpi=300, bbox_inches='tight')
    
    plt.show()
    plt.close()

# Selected complex features

In [ ]:
selected_complex_for_feature_comparison = {
    "kinetochore": {
        "inner kinetochore": ["cnp1","cnl2", "fta1", "fta2", "fta3", "fta4", "fta6", "fta7", "mal2", "mis15", "mis17", "mis6", "sim4", "cnp20", "wip1", "mhf1", "mhf2"],
        "KMN complex (outer kinetochore)": ["spc7", "sos7", "mis12", "mis13", "mis14", "nnf1", "ndc80", "spc25", "spc24", "nuf2"],
        "DASH complex (outer kinetochore)": ["ask1", "dad1", "dad2", "dad3", "dad4", "dad5", "dam1", "duo1", "spc19", "spc34"],
    },
    "vesicle-related complexes": {
        "SNARE complex": ["bet1", "bos1", "fsv1", "gos1", "pep12", "psy1", "sec20", "sec22", "sec9", "sed5", "sft1", "syb1", "tlg1", "tlg2", "ufe1", "use1", "vsl1", "vti1", "ykt6"],
        "exocyst complex": ["exo70", "exo8", "sec3", "sec5", "sec6", "sec8", "sec10", "sec15"],
        "TRAPP complex": ["bet3", "bet5", "trs23", "trs31", "trs33", "trs65", "trs120", "trs130", "trs8501", "trs8502", "tca17"],
    },
    "mitochondrion":{
        "protein import into mitochondrial matrix (PAM + TIM23 + TOM)": ["mcp60", "mge1", "mgr2", "mom14", "pam16", "pam17", "pam18", "ssc1", "tim17", "tim21", "tim23", "tim44", "tim50", "tom20", "tom22", "tom5", "tom6", "tom7", "xdj1", "zim17"], # PAM + TIM23 + TOM
        "mitochondrial translation": [
            "cbp7", "cox1101", "cox1102", "msr1", "SPAC29E6.06c", "grs1", "hrs1", "tac1", "gfm1", "ppr10",
            "gta1", "gta2", "gta3", "mse1", "ism1", "SPAC4G8.09", "msk1", "mdm38", "msm1", "fmt1",
            "mtq1", "pth4", "pth1", "pth3", "msf1", "ppr4", "SPBC24C6.03", "mrpl1", "mrp11", "mrpl19",
            "mrpl12", "mrpl23", "mrpl38", "mrpl10", "mrpl16", "mrpl8", "img1", "rml2", "aco2", "mrpl22",
            "mrp20", "mrpl40", "mrp7", "mrpl24", "mrpl4", "mrpl9", "mrpl33", "tam9", "mrpl32", "mrpl39",
            "mrx14", "new15", "rtc6", "mrpl35", "yml6", "mrpl28", "mrpl27", "mrpl51", "mrpl3", "mrpl17",
            "img2", "mrpl7", "mrpl50", "mug178", "mrp51", "mrpl44", "mrpl37", "mrpl15", "mrpl20", "mrpl25",
            "mrpl6", "mrpl31", "mhr1", "SPCC777.17c", "mrp49", "rsm10", "mrps18", "mrps12", "sws2",
            "mrp2", "mrps28", "mrps16", "mrps17", "rsm18", "rsm19", "mrp4", "mrp21", "rsm25", "mrps26",
            "rsm23", "var1", "rsm27", "rsm24", "mrp10", "cox24", "nam9", "fyv4", "SPBC3H7.04", "SPBC16A3.14",
            "bot1", "snr1", "mrp5", "mrp17", "rsm7", "mrps8", "mrps9", "rrf1", "gfm2", "dia4",
            "SPAC24C9.09", "mrh5", "tsf1", "tuf1", "guf1", "mti2", "mti3", "mrf1", "cbp8", "mtf2",
            "sls1", "mpa1", "SPCC576.06c", "vrs2"
        ],
        "iron-sulfur cluster": ["frp1","fip1","fio1","fet4","mmt1","fxn1","nfs1","isd11","isu1","arh1","etp1","ssc1","jac1","grx5","isa1","isa2","iba57","atm1","dre2","tah18","SPAC806.02c","nar1","mms19","rli1","grx4","fra2","fep1","php4"],
        "cytosolic and nuclear 4Fe-4S cluster assembly": ["tah18", "dre2", "SPAC806.02c", "nar1", "mms19", "rli1"],
        "TCA cycle": ["aco1", "aco2", "cit1", "fum1", "idh1", "idh2", "idp1", "kgd1", "kgd2", "kgd4", "lsc2", "mdh1", "sca1", "sdh1", "sdh2", "sdh3", "sdh4"],
        "oxidative phosphorylation": ["cob1", "cyc1","cox12","cox1","cox2","cox3","cox4","cox5","cox6","cox13","cox7","cox9","cox8","cyt1","nde1","atp1","atp2","atp16","atp15","tim11","atp17","atp20","atp14","atp18","atp19","atp3","atp6","atp8","atp9","atp7","atp5","atp4","ndi1","inh1","sdh3","sdh2","sdh1","qcr2","rip1","qcr7","qcr8","qcr6","qcr9","qcr10"],
    }
}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator, FixedLocator, NullLocator
from scipy import stats

# 去除基因名为 NaN 的空行
plot_data = feature_and_targets.dropna(subset=["gene_name"])

key_features = [
    "mean_EMM_Proliferating_Cell_RNA_Abundance",
    "mRNA_half_life_minutes",
    "mRNA_synthesis_rate_per_minute",
    "copies_per_cell_EMM_Proliferating_Cell",
    "protein_half_life_minutes",
    "evolutionary_rate"
]

feature_info = {
    "mean_EMM_Proliferating_Cell_RNA_Abundance": {"title": "RNA Abundance", "unit": "Copies / Cell"},
    "mRNA_half_life_minutes": {"title": "mRNA Half-life", "unit": "Minutes"},
    "mRNA_synthesis_rate_per_minute": {"title": "mRNA Synthesis Rate", "unit": "Molecules / Minute"},
    "copies_per_cell_EMM_Proliferating_Cell": {"title": "Protein Copies per Cell", "unit": "Copies / Cell"},
    "protein_half_life_minutes": {"title": "Protein Half-life", "unit": "Minutes"},
    "evolutionary_rate": {"title": "Evolutionary Rate", "unit": "Substitutions / Site"}
}

base_colors = [
    "#dd8369", "#6b99df", "#98a64e", "#64af6d", "#a78bd9", 
    "#d57fbd", "#c4954b", "#4bb29c", "#e0788f", "#4aadce"
]

# -------- 将嵌套的字典展平，分配分类专属颜色，并记录大类归属 --------
flattened_complexes = []
for cat_idx, (cat_name, complexes_dict) in enumerate(selected_complex_for_feature_comparison.items()):
    cat_color = base_colors[cat_idx % len(base_colors)] # 同一个 category 共用一个颜色
    for comp_name, genes in complexes_dict.items():
        flattened_complexes.append({
            "category": cat_name,
            "complex": comp_name,
            "genes": genes,
            "color": cat_color
        })

# 不反转，直接采用字典原有的分类顺序
sorted_complexes = flattened_complexes

if not plot_data.empty:
    fig, axes = plt.subplots(2, 3, figsize=(24, 16))
    axes = axes.flatten()
    
    dark_gray = '#555555'
    
    for i, feature in enumerate(key_features):
        ax = axes[i]
        
        log_transformed_data = [] 
        raw_data_for_test = []    
        y_labels = []
        colors = []
        categories_for_test = [] 
        
        for comp_info in sorted_complexes:
            genes = comp_info["genes"]
            comp_name = comp_info["complex"]
            category = comp_info["category"]
            box_color = comp_info["color"]
            
            raw_vals = plot_data[plot_data["gene_name"].isin(genes)][feature].dropna()
            
            if i != 5: # 前 5 个需要 log 转换
                raw_vals = raw_vals[raw_vals > 0] 
                plot_vals = np.log10(raw_vals)
            else:
                plot_vals = raw_vals 
                
            if len(raw_vals) > 0:
                raw_data_for_test.append(raw_vals)
                log_transformed_data.append(plot_vals)
                colors.append(box_color)
                categories_for_test.append(category)
                
                # 追加 n=x
                y_labels.append(f"{comp_name} (n={len(raw_vals)})")
        
        if log_transformed_data:
            positions = np.arange(len(log_transformed_data))
            
            bp = ax.boxplot(log_transformed_data, positions=positions, patch_artist=True,
                           vert=False, showfliers=False,
                           boxprops=dict(linewidth=1.5, color=dark_gray),
                           medianprops=dict(color=dark_gray, linewidth=1.5, solid_capstyle='butt'),
                           whiskerprops=dict(linewidth=1.5, color=dark_gray),
                           capprops=dict(linewidth=1.5, color=dark_gray))
            
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_edgecolor(dark_gray)
                patch.set_alpha(0.8)
            
            # --------- 添加组间 p-value ---------
            for j in range(len(raw_data_for_test) - 1):
                if categories_for_test[j] != categories_for_test[j + 1]:
                    continue
                
                stat, p_val = stats.mannwhitneyu(raw_data_for_test[j], raw_data_for_test[j + 1], alternative='two-sided')
                
                sig = f'p={p_val:.1e}' if p_val < 0.001 else f'p={p_val:.3f}'
                text_color = '#8A2A44' if p_val < 0.05 else 'black'
                font_weight = 'bold' if p_val < 0.05 else 'normal'
                
                y1, y2 = j + 0.1, j + 1 - 0.1
                x_bracket, x_text = 1.02, 1.05
                
                ax.plot([x_bracket, x_text, x_text, x_bracket], [y1, y1, y2, y2], 
                        transform=ax.get_yaxis_transform(), color='black', lw=1.2, clip_on=False)
                ax.text(x_text + 0.01, (y1 + y2) / 2, sig, transform=ax.get_yaxis_transform(), 
                        ha='left', va='center', fontsize=12, color=text_color, fontweight=font_weight, clip_on=False)
            
            ax.set_yticks(positions)
            
            # 反转 Y 轴方向
            ax.invert_yaxis()
            
        # --------- 取消 wrap 单行显示，并修改每行 label 颜色跟随箱线图类别变色 ---------
        ax.set_yticklabels(y_labels, fontsize=12, fontweight='bold')
        for tick_label, color in zip(ax.get_yticklabels(), colors):
            tick_label.set_color(color)
            
        ax.yaxis.set_minor_locator(NullLocator())
        
        ax.set_title(feature_info[feature]["title"], fontsize=16, fontweight='bold', pad=10)
        ax.set_xlabel(feature_info[feature]["unit"], fontsize=14, fontweight='bold')
        
        ax.grid(True, axis='x', which='major', alpha=0.3, linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=14, labelcolor='black', 
                       length=10, width=2, color='gray', direction='out')

        # Log Scale 视觉外观
        if i != 5:
            ax.xaxis.set_major_locator(MultipleLocator(base=1.0))
            
            minor_ticks = []
            for base_val in range(-5, 10):
                for step in range(2, 10):
                    minor_ticks.append(base_val + np.log10(step))
            ax.xaxis.set_minor_locator(FixedLocator(minor_ticks))
            
            def format_log_ticks(val, pos):
                real_val = 10**val
                if real_val >= 1000 or real_val <= 0.01:
                    return f"$10^{{{int(val)}}}$"
                elif real_val >= 1:
                    return f"{int(real_val)}"
                else:
                    return f"{real_val:.1f}" if real_val == 0.1 else f"{real_val:.2g}"

            ax.xaxis.set_major_formatter(FuncFormatter(format_log_ticks))
            ax.tick_params(axis='x', which='minor', length=5, width=1, color='gray', direction='out')
            ax.tick_params(axis='x', which='major', length=10, width=2, color='gray', direction='out')
            
        else:
            # 最后一个特殊处理防止坐标轴颜色被改变覆盖
            ax.tick_params(axis='x', which='major', labelsize=14, labelcolor='black', 
                           length=10, width=2, color='gray', direction='out')
    
    # 取消文字换行后可能需要预留更多的左侧边距调整紧凑度
    plt.tight_layout(w_pad=8) 
    
    if 'cfg' in locals() and hasattr(cfg, 'output_dir'):
        plt.savefig(cfg.output_dir / "complexes_features_boxplot.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(cfg.output_dir / "complexes_features_boxplot.svg", dpi=300, bbox_inches='tight')
    
    plt.show()
    plt.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator, FixedLocator, NullLocator
from scipy import stats
import seaborn as sns
import textwrap

# 去除基因名为 NaN 的空行
plot_data = feature_and_targets.dropna(subset=["gene_name"])

key_features = [
    "mean_EMM_Proliferating_Cell_RNA_Abundance",
    "mRNA_half_life_minutes",
    "mRNA_synthesis_rate_per_minute",
    "copies_per_cell_EMM_Proliferating_Cell",
    "protein_half_life_minutes",
    "evolutionary_rate"
]

feature_info = {
    "mean_EMM_Proliferating_Cell_RNA_Abundance": {"title": "RNA Abundance", "unit": "Copies / Cell"},
    "mRNA_half_life_minutes": {"title": "mRNA Half-life", "unit": "Minutes"},
    "mRNA_synthesis_rate_per_minute": {"title": "mRNA Synthesis Rate", "unit": "Molecules / Minute"},
    "copies_per_cell_EMM_Proliferating_Cell": {"title": "Protein Copies per Cell", "unit": "Copies / Cell"},
    "protein_half_life_minutes": {"title": "Protein Half-life", "unit": "Minutes"},
    "evolutionary_rate": {"title": "Evolutionary Rate", "unit": "Substitutions / Site"}
}

base_colors = [
    "#dd8369", "#6b99df", "#98a64e", "#64af6d", "#a78bd9", 
    "#d57fbd", "#c4954b", "#4bb29c", "#e0788f", "#4aadce"
]

# -------- 将嵌套的字典展平，分配分类专属颜色，并记录大类归属 --------
flattened_complexes = []
for cat_idx, (cat_name, complexes_dict) in enumerate(selected_complex_for_feature_comparison.items()):
    cat_color = base_colors[cat_idx % len(base_colors)] # 同一个 category 共用一个颜色
    for comp_name, genes in complexes_dict.items():
        flattened_complexes.append({
            "category": cat_name,
            "complex": comp_name,
            "genes": genes,
            "color": cat_color
        })

# 翻转顺序，使得原字典里排在大类最顶上的 complex，在绘图时 Y 轴坐标最高（位于视觉最顶部）
sorted_complexes = list(reversed(flattened_complexes))

if not plot_data.empty:
    fig, axes = plt.subplots(2, 3, figsize=(24, 16))
    axes = axes.flatten()
    
    dark_gray = '#555555'
    
    for i, feature in enumerate(key_features):
        ax = axes[i]
        
        log_transformed_data = [] 
        raw_data_for_test = []    
        y_labels = []
        colors = []
        categories_for_test = [] # 记录当前有效数据的 category，用于阻断跨类的 P 值计算
        
        for comp_info in sorted_complexes:
            genes = comp_info["genes"]
            comp_name = comp_info["complex"]
            category = comp_info["category"]
            box_color = comp_info["color"]
            
            # 提取数据
            raw_vals = plot_data[plot_data["gene_name"].isin(genes)][feature].dropna()
            
            if i != 5: # 前 5 个特征需要 log 转换绘制
                raw_vals = raw_vals[raw_vals > 0] 
                plot_vals = np.log10(raw_vals)
            else:
                plot_vals = raw_vals 
                
            if len(raw_vals) > 0:
                raw_data_for_test.append(raw_vals)
                log_transformed_data.append(plot_vals)
                colors.append(box_color)
                categories_for_test.append(category)
                
                # 追加 n=x 大小到该特征的 Y 显示标签中 
                y_labels.append(f"{comp_name} (n={len(raw_vals)})")
        
        if log_transformed_data:
            positions = np.arange(len(log_transformed_data))
            
            bp = ax.boxplot(log_transformed_data, positions=positions, patch_artist=True,
                           vert=False, showfliers=False,
                           boxprops=dict(linewidth=1.5, color=dark_gray),
                           medianprops=dict(color=dark_gray, linewidth=1.5, solid_capstyle='butt'),
                           whiskerprops=dict(linewidth=1.5, color=dark_gray),
                           capprops=dict(linewidth=1.5, color=dark_gray))
            
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_edgecolor(dark_gray)
                patch.set_alpha(0.6) 
            
            x_swarm = []
            y_swarm = []
            for j, vals in enumerate(log_transformed_data):
                x_swarm.extend(vals)
                y_swarm.extend([j] * len(vals))
                
            sns.swarmplot(x=x_swarm, y=y_swarm, orient='h', size=4, color=".2", ax=ax, alpha=0.8, zorder=3)
            
            # --------- 添加组间 p-value，增加跨类截断 ---------
            for j in range(len(raw_data_for_test) - 1):
                # ★ 只有当上下两个复合物同属于一个 category 时，才计算与绘制连接线
                if categories_for_test[j] != categories_for_test[j + 1]:
                    continue
                
                stat, p_val = stats.mannwhitneyu(raw_data_for_test[j], raw_data_for_test[j + 1], alternative='two-sided')
                
                sig = f'p={p_val:.1e}' if p_val < 0.001 else f'p={p_val:.3f}'
                text_color = '#8A2A44' if p_val < 0.05 else 'black'
                font_weight = 'bold' if p_val < 0.05 else 'normal'
                
                y1, y2 = j + 0.1, j + 1 - 0.1
                x_bracket, x_text = 1.02, 1.05
                
                ax.plot([x_bracket, x_text, x_text, x_bracket], [y1, y1, y2, y2], 
                        transform=ax.get_yaxis_transform(), color='black', lw=1.2, clip_on=False)
                ax.text(x_text + 0.01, (y1 + y2) / 2, sig, transform=ax.get_yaxis_transform(), 
                        ha='left', va='center', fontsize=12, color=text_color, fontweight=font_weight, clip_on=False)
            
            ax.set_yticks(positions)
            
        # 换行处理 (含刚刚新接的 n=x)
        wrapped_y_labels = [textwrap.fill(label, width=28) for label in y_labels]
        ax.set_yticklabels(wrapped_y_labels, fontsize=12, fontweight='bold')
        ax.yaxis.set_minor_locator(NullLocator())
        
        ax.set_title(feature_info[feature]["title"], fontsize=16, fontweight='bold', pad=10)
        ax.set_xlabel(feature_info[feature]["unit"], fontsize=14, fontweight='bold')
        
        ax.grid(True, axis='x', which='major', alpha=0.3, linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=14, labelcolor='black', 
                       length=10, width=2, color='gray', direction='out')

        # Log Scale 视觉外观
        if i != 5:
            ax.xaxis.set_major_locator(MultipleLocator(base=1.0))
            
            minor_ticks = []
            for base_val in range(-5, 10):
                for step in range(2, 10):
                    minor_ticks.append(base_val + np.log10(step))
            ax.xaxis.set_minor_locator(FixedLocator(minor_ticks))
            
            def format_log_ticks(val, pos):
                real_val = 10**val
                if real_val >= 1000 or real_val <= 0.01:
                    return f"$10^{{{int(val)}}}$"
                elif real_val >= 1:
                    return f"{int(real_val)}"
                else:
                    return f"{real_val:.1f}" if real_val == 0.1 else f"{real_val:.2g}"

            ax.xaxis.set_major_formatter(FuncFormatter(format_log_ticks))
            ax.tick_params(axis='x', which='minor', length=5, width=1, color='gray', direction='out')
            ax.tick_params(axis='x', which='major', length=10, width=2, color='gray', direction='out')
    
    plt.tight_layout(w_pad=8) 
    
    if 'cfg' in locals() and hasattr(cfg, 'output_dir'):
        plt.savefig(cfg.output_dir / "complexes_features_boxplot.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(cfg.output_dir / "complexes_features_boxplot.svg", dpi=300, bbox_inches='tight')
    
    plt.show()
    plt.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator, FixedLocator, NullLocator
from scipy import stats
import textwrap

# 去除基因名为 NaN 的空行
plot_data = feature_and_targets.dropna(subset=["gene_name"])

key_features = [
    "mean_EMM_Proliferating_Cell_RNA_Abundance",
    "mRNA_half_life_minutes",
    "mRNA_synthesis_rate_per_minute",
    "copies_per_cell_EMM_Proliferating_Cell",
    "protein_half_life_minutes",
    "evolutionary_rate"
]

feature_info = {
    "mean_EMM_Proliferating_Cell_RNA_Abundance": {"title": "RNA Abundance", "unit": "Copies / Cell"},
    "mRNA_half_life_minutes": {"title": "mRNA Half-life", "unit": "Minutes"},
    "mRNA_synthesis_rate_per_minute": {"title": "mRNA Synthesis Rate", "unit": "Molecules / Minute"},
    "copies_per_cell_EMM_Proliferating_Cell": {"title": "Protein Copies per Cell", "unit": "Copies / Cell"},
    "protein_half_life_minutes": {"title": "Protein Half-life", "unit": "Minutes"},
    "evolutionary_rate": {"title": "Evolutionary Rate", "unit": "Substitutions / Site"}
}

base_colors = [
    "#dd8369", "#6b99df", "#98a64e", "#64af6d", "#a78bd9", 
    "#d57fbd", "#c4954b", "#4bb29c", "#e0788f", "#4aadce"
]

# -------- 将嵌套的字典展平，分配分类专属颜色，并记录大类归属 --------
flattened_complexes = []
for cat_idx, (cat_name, complexes_dict) in enumerate(selected_complex_for_feature_comparison.items()):
    cat_color = base_colors[cat_idx % len(base_colors)] # 同一个 category 共用一个颜色
    for comp_name, genes in complexes_dict.items():
        flattened_complexes.append({
            "category": cat_name,
            "complex": comp_name,
            "genes": genes,
            "color": cat_color
        })

# 去除原本的反转逻辑，直接采用字典原有顺序
sorted_complexes = flattened_complexes

if not plot_data.empty:
    fig, axes = plt.subplots(2, 3, figsize=(24, 16))
    axes = axes.flatten()
    
    dark_gray = '#555555'
    
    for i, feature in enumerate(key_features):
        ax = axes[i]
        
        log_transformed_data = [] 
        raw_data_for_test = []    
        y_labels = []
        colors = []
        categories_for_test = [] 
        
        for comp_info in sorted_complexes:
            genes = comp_info["genes"]
            comp_name = comp_info["complex"]
            category = comp_info["category"]
            box_color = comp_info["color"]
            
            raw_vals = plot_data[plot_data["gene_name"].isin(genes)][feature].dropna()
            
            if i != 5: # 前 5 个需要 log 转换
                raw_vals = raw_vals[raw_vals > 0] 
                plot_vals = np.log10(raw_vals)
            else:
                plot_vals = raw_vals 
                
            if len(raw_vals) > 0:
                raw_data_for_test.append(raw_vals)
                log_transformed_data.append(plot_vals)
                colors.append(box_color)
                categories_for_test.append(category)
                
                # 追加 n=x
                y_labels.append(f"{comp_name} (n={len(raw_vals)})")
        
        if log_transformed_data:
            positions = np.arange(len(log_transformed_data))
            
            bp = ax.boxplot(log_transformed_data, positions=positions, patch_artist=True,
                           vert=False, showfliers=False,
                           boxprops=dict(linewidth=1.5, color=dark_gray),
                           medianprops=dict(color=dark_gray, linewidth=1.5, solid_capstyle='butt'),
                           whiskerprops=dict(linewidth=1.5, color=dark_gray),
                           capprops=dict(linewidth=1.5, color=dark_gray))
            
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_edgecolor(dark_gray)
                patch.set_alpha(0.8)
            
            # --------- 添加组间 p-value ---------
            for j in range(len(raw_data_for_test) - 1):
                if categories_for_test[j] != categories_for_test[j + 1]:
                    continue
                
                stat, p_val = stats.mannwhitneyu(raw_data_for_test[j], raw_data_for_test[j + 1], alternative='two-sided')
                
                sig = f'p={p_val:.1e}' if p_val < 0.001 else f'p={p_val:.3f}'
                text_color = '#8A2A44' if p_val < 0.05 else 'black'
                font_weight = 'bold' if p_val < 0.05 else 'normal'
                
                y1, y2 = j + 0.1, j + 1 - 0.1
                x_bracket, x_text = 1.02, 1.05
                
                ax.plot([x_bracket, x_text, x_text, x_bracket], [y1, y1, y2, y2], 
                        transform=ax.get_yaxis_transform(), color='black', lw=1.2, clip_on=False)
                ax.text(x_text + 0.01, (y1 + y2) / 2, sig, transform=ax.get_yaxis_transform(), 
                        ha='left', va='center', fontsize=12, color=text_color, fontweight=font_weight, clip_on=False)
            
            ax.set_yticks(positions)
            
            # 反转 Y 轴，确保位置 0 (字典第一项) 出现在图表最上方
            ax.invert_yaxis()
            
        # 换行处理
        wrapped_y_labels = [textwrap.fill(label, width=28) for label in y_labels]
        ax.set_yticklabels(wrapped_y_labels, fontsize=12, fontweight='bold')
        ax.yaxis.set_minor_locator(NullLocator())
        
        ax.set_title(feature_info[feature]["title"], fontsize=16, fontweight='bold', pad=10)
        ax.set_xlabel(feature_info[feature]["unit"], fontsize=14, fontweight='bold')
        
        ax.grid(True, axis='x', which='major', alpha=0.3, linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=14, labelcolor='black', 
                       length=10, width=2, color='gray', direction='out')

        # Log Scale 视觉外观
        if i != 5:
            ax.xaxis.set_major_locator(MultipleLocator(base=1.0))
            
            minor_ticks = []
            for base_val in range(-5, 10):
                for step in range(2, 10):
                    minor_ticks.append(base_val + np.log10(step))
            ax.xaxis.set_minor_locator(FixedLocator(minor_ticks))
            
            def format_log_ticks(val, pos):
                real_val = 10**val
                if real_val >= 1000 or real_val <= 0.01:
                    return f"$10^{{{int(val)}}}$"
                elif real_val >= 1:
                    return f"{int(real_val)}"
                else:
                    return f"{real_val:.1f}" if real_val == 0.1 else f"{real_val:.2g}"

            ax.xaxis.set_major_formatter(FuncFormatter(format_log_ticks))
            ax.tick_params(axis='x', which='minor', length=5, width=1, color='gray', direction='out')
            ax.tick_params(axis='x', which='major', length=10, width=2, color='gray', direction='out')
    
    plt.tight_layout(w_pad=8) 
    
    if 'cfg' in locals() and hasattr(cfg, 'output_dir'):
        plt.savefig(cfg.output_dir / "complexes_features_boxplot.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(cfg.output_dir / "complexes_features_boxplot.svg", dpi=300, bbox_inches='tight')
    
    plt.show()
    plt.close()